In [ ]:
# ============================================================
# CELL 1
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------
# Standard Library
# ------------------------------------------------------------

import os
import json
import random
import shutil
import time

# ------------------------------------------------------------
# Numerical Computing
# ------------------------------------------------------------

import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm

from numpy.linalg import inv

# ------------------------------------------------------------
# PyTorch
# ------------------------------------------------------------

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torchvision import transforms

from torchvision.models import (
    mobilenet_v3_small,
    MobileNet_V3_Small_Weights
)

# ------------------------------------------------------------
# Scikit-Learn
# ------------------------------------------------------------

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier
)

from sklearn.linear_model import (
    RidgeClassifier,
    LogisticRegression
)

# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------

from IPython.display import display

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Mounted at /content/drive


In [ ]:
# ============================================================
# CELL 2
# DATASET SETUP + ANALYSIS
# ============================================================

# ------------------------------------------------------------
# Dataset Paths
# ------------------------------------------------------------

DATASET_ROOT = (
    "/content/drive/MyDrive/TinyBayes/amini_dataset"
)

BASE_DIR = os.path.join(
    DATASET_ROOT,
    "dataset"
)

CSV_PATH_TRAIN = os.path.join(
    DATASET_ROOT,
    "Train.csv"
)

IMAGE_DIR_TRAIN = os.path.join(
    BASE_DIR,
    "images",
    "train"
)

LABEL_DIR_TRAIN = os.path.join(
    BASE_DIR,
    "labels",
    "train"
)

SAVE_DIR = (
    "/content/drive/MyDrive/TinyBayes/run_2000"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Verify Dataset
# ------------------------------------------------------------

print("=" * 60)
print("VERIFYING DATASET")
print("=" * 60)

paths = {
    "Train CSV"     : CSV_PATH_TRAIN,
    "Train Images"  : IMAGE_DIR_TRAIN,
    "Train Labels"  : LABEL_DIR_TRAIN,
    "Output Folder" : SAVE_DIR
}

for name, path in paths.items():

    print(
        f"{name:<15}:",
        os.path.exists(path)
    )

# ------------------------------------------------------------
# Load CSV
# ------------------------------------------------------------

df_train = pd.read_csv(
    CSV_PATH_TRAIN
)

# ------------------------------------------------------------
# Dataset Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print(
    "Training Annotation Rows:",
    len(df_train)
)

print(
    "\nTraining Shape:",
    df_train.shape
)

print(
    "\nTraining Columns:"
)

print(
    df_train.columns.tolist()
)

print(
    "\nAnnotation-Level Class Distribution:"
)

print(
    df_train["class"].value_counts()
)

print(
    "\nUnique Classes:"
)

print(
    sorted(df_train["class"].unique())
)

print(
    "\nUnique Training Images:",
    df_train["Image_ID"].nunique()
)

print("\nFirst Five Rows:")

display(
    df_train.head()
)

# ------------------------------------------------------------
# Label File Inspection
# ------------------------------------------------------------

label_files = sorted(
    os.listdir(
        LABEL_DIR_TRAIN
    )
)

print(
    "\nNumber of Label Files:",
    len(label_files)
)

if label_files:

    sample_label = os.path.join(
        LABEL_DIR_TRAIN,
        label_files[0]
    )

    print(
        "\nExample Label File:",
        label_files[0]
    )

    with open(sample_label) as f:

        print("\nContents:")
        print(f.read())

# ------------------------------------------------------------
# Image-Level Analysis
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("IMAGE-LEVEL ANALYSIS")
print("=" * 60)

image_class_counts = (
    df_train
    .groupby("Image_ID")["class"]
    .nunique()
)

ambiguous_images = image_class_counts[
    image_class_counts > 1
]

single_label_images = image_class_counts[
    image_class_counts == 1
]

print(
    "Unique Images:",
    len(image_class_counts)
)

print(
    "Single-Label Images:",
    len(single_label_images)
)

print(
    "Multi-Label Images:",
    len(ambiguous_images)
)

if len(ambiguous_images) > 0:

    print("\nExample Ambiguous Images:")

    display(
        df_train[
            df_train["Image_ID"].isin(
                ambiguous_images.index[:5]
            )
        ].sort_values("Image_ID")
    )

# ------------------------------------------------------------
# Save Statistics
# ------------------------------------------------------------

NUM_CLASSES = df_train["class"].nunique()

NUM_UNIQUE_IMAGES = (
    df_train["Image_ID"].nunique()
)

CLASS_NAMES = sorted(
    df_train["class"].unique()
)

VERIFYING DATASET
Train CSV      : True
Train Images   : True
Train Labels   : True
Output Folder  : True

DATASET SUMMARY
Training Annotation Rows: 9792

Training Shape: (9792, 9)

Training Columns:
['Image_ID', 'class', 'confidence', 'ymin', 'xmin', 'ymax', 'xmax', 'class_id', 'ImagePath']

Annotation-Level Class Distribution:
class
healthy        4280
cssvd          3241
anthracnose    2271
Name: count, dtype: int64

Unique Classes:
['anthracnose', 'cssvd', 'healthy']

Unique Training Images: 5529

First Five Rows:


,Image_ID,class,confidence,ymin,xmin,ymax,xmax,class_id,ImagePath
0,ID_nBgcAR.jpg,healthy,1.0,75.0,15.0,162.0,195.0,2,dataset/images/train/ID_nBgcAR.jpg
1,ID_nBgcAR.jpg,healthy,1.0,58.0,1.0,133.0,171.0,2,dataset/images/train/ID_nBgcAR.jpg
2,ID_nBgcAR.jpg,healthy,1.0,42.0,29.0,377.0,349.0,2,dataset/images/train/ID_nBgcAR.jpg
3,ID_Kw2v8A.jpg,healthy,1.0,112.0,124.0,404.0,341.0,2,dataset/images/train/ID_Kw2v8A.jpg
4,ID_Kw2v8A.jpg,healthy,1.0,148.0,259.0,413.0,412.0,2,dataset/images/train/ID_Kw2v8A.jpg



Number of Label Files: 5529

Example Label File: ID_A0o1FA.txt

Contents:
1 0.335483 0.682912 0.670966 0.605903


IMAGE-LEVEL ANALYSIS
Unique Images: 5529
Single-Label Images: 5523
Multi-Label Images: 6

Example Ambiguous Images:


,Image_ID,class,confidence,ymin,xmin,ymax,xmax,class_id,ImagePath
4010,ID_BRJQ51.JPG,healthy,1.0,26.0,73.0,2809.0,1647.0,2,dataset/images/train/ID_BRJQ51.JPG
5712,ID_BRJQ51.JPG,cssvd,1.0,129.0,260.0,2877.0,1122.0,1,dataset/images/train/ID_BRJQ51.JPG
3632,ID_I9gkRe.JPG,healthy,1.0,20.0,54.0,3264.0,2076.0,2,dataset/images/train/ID_I9gkRe.JPG
3633,ID_I9gkRe.JPG,healthy,1.0,0.0,1731.0,1834.0,2448.0,2,dataset/images/train/ID_I9gkRe.JPG
5616,ID_I9gkRe.JPG,cssvd,1.0,75.0,260.0,3264.0,1970.0,1,dataset/images/train/ID_I9gkRe.JPG
3992,ID_nRsOlR.JPG,healthy,1.0,20.0,21.0,2620.0,1146.0,2,dataset/images/train/ID_nRsOlR.JPG
3993,ID_nRsOlR.JPG,healthy,1.0,0.0,1022.0,2919.0,2448.0,2,dataset/images/train/ID_nRsOlR.JPG
5629,ID_nRsOlR.JPG,cssvd,1.0,0.0,993.0,2633.0,1991.0,1,dataset/images/train/ID_nRsOlR.JPG
4328,ID_r1LWke.JPG,healthy,1.0,13.0,0.0,3264.0,1939.0,2,dataset/images/train/ID_r1LWke.JPG
5723,ID_r1LWke.JPG,cssvd,1.0,75.0,0.0,3264.0,1699.0,1,dataset/images/train/ID_r1LWke.JPG


In [ ]:
# ============================================================
# CELL 3
# CREATE IMAGE-LEVEL TRAIN / VALIDATION SPLITS
# ============================================================

TRAIN_SIZE = 2000
VAL_SIZE = 500

TRAIN_SPLIT_PATH = os.path.join(
    SAVE_DIR,
    "train_2000_unique_images.csv"
)

VAL_SPLIT_PATH = os.path.join(
    SAVE_DIR,
    "val_500_unique_images.csv"
)

# ============================================================
# LOAD EXISTING SPLITS
# ============================================================

if (
    os.path.exists(TRAIN_SPLIT_PATH)
    and
    os.path.exists(VAL_SPLIT_PATH)
):

    print("=" * 60)
    print("LOADING EXISTING SPLITS")
    print("=" * 60)

    df_train_balanced = pd.read_csv(
        TRAIN_SPLIT_PATH
    )

    df_validation = pd.read_csv(
        VAL_SPLIT_PATH
    )

# ============================================================
# CREATE NEW SPLITS
# ============================================================

else:

    print("=" * 60)
    print("CREATING TRAIN / VALIDATION SPLITS")
    print("=" * 60)

    # --------------------------------------------------------
    # Remove ambiguous images
    # --------------------------------------------------------

    image_class_count = (
        df_train
        .groupby("Image_ID")["class"]
        .nunique()
    )

    ambiguous_images = set(
        image_class_count[
            image_class_count > 1
        ].index
    )

    print(
        "Ambiguous Images:",
        len(ambiguous_images)
    )

    clean_df = df_train[
        ~df_train["Image_ID"].isin(
            ambiguous_images
        )
    ].copy()

    # --------------------------------------------------------
    # One row per image
    # --------------------------------------------------------

    unique_images = (
        clean_df[
            ["Image_ID", "class"]
        ]
        .drop_duplicates(
            subset="Image_ID"
        )
        .reset_index(drop=True)
    )

    print(
        "Clean Images:",
        len(unique_images)
    )

    # --------------------------------------------------------
    # Verify enough samples exist
    # --------------------------------------------------------

    train_per_class = TRAIN_SIZE // NUM_CLASSES
    val_per_class = VAL_SIZE // NUM_CLASSES

    for cls in CLASS_NAMES:

        count = (
            unique_images["class"] == cls
        ).sum()

        assert count >= (
            train_per_class +
            val_per_class
        ), f"Not enough samples for {cls}"

    # --------------------------------------------------------
    # Balanced Train Split
    # --------------------------------------------------------

    train_parts = []

    for cls in CLASS_NAMES:

        cls_images = unique_images[
            unique_images["class"] == cls
        ]

        sampled = cls_images.sample(
            n=train_per_class,
            random_state=SEED
        )

        train_parts.append(sampled)

    df_train_balanced = (
        pd.concat(train_parts)
        .sample(
            frac=1,
            random_state=SEED
        )
        .reset_index(drop=True)
    )

    remaining = (
        TRAIN_SIZE -
        len(df_train_balanced)
    )

    if remaining > 0:

        pool = unique_images[
            ~unique_images["Image_ID"].isin(
                df_train_balanced["Image_ID"]
            )
        ]

        extra = pool.sample(
            n=remaining,
            random_state=SEED
        )

        df_train_balanced = (
            pd.concat(
                [
                    df_train_balanced,
                    extra
                ]
            )
            .sample(
                frac=1,
                random_state=SEED
            )
            .reset_index(drop=True)
        )

    # --------------------------------------------------------
    # Balanced Validation Split
    # --------------------------------------------------------

    remaining_images = unique_images[
        ~unique_images["Image_ID"].isin(
            df_train_balanced["Image_ID"]
        )
    ]

    validation_parts = []

    for cls in CLASS_NAMES:

        cls_images = remaining_images[
            remaining_images["class"] == cls
        ]

        sampled = cls_images.sample(
            n=val_per_class,
            random_state=SEED
        )

        validation_parts.append(sampled)

    df_validation = (
        pd.concat(validation_parts)
        .sample(
            frac=1,
            random_state=SEED
        )
        .reset_index(drop=True)
    )

    remaining = (
        VAL_SIZE -
        len(df_validation)
    )

    if remaining > 0:

        pool = remaining_images[
            ~remaining_images["Image_ID"].isin(
                df_validation["Image_ID"]
            )
        ]

        extra = pool.sample(
            n=remaining,
            random_state=SEED
        )

        df_validation = (
            pd.concat(
                [
                    df_validation,
                    extra
                ]
            )
            .sample(
                frac=1,
                random_state=SEED
            )
            .reset_index(drop=True)
        )

    # --------------------------------------------------------
    # Save Splits
    # --------------------------------------------------------

    df_train_balanced.to_csv(
        TRAIN_SPLIT_PATH,
        index=False
    )

    df_validation.to_csv(
        VAL_SPLIT_PATH,
        index=False
    )

    print("\nSaved splits:")
    print(TRAIN_SPLIT_PATH)
    print(VAL_SPLIT_PATH)

# ============================================================
# VERIFY SPLITS
# ============================================================

train_ids = set(
    df_train_balanced["Image_ID"]
)

validation_ids = set(
    df_validation["Image_ID"]
)

overlap = train_ids.intersection(
    validation_ids
)

print("\n" + "=" * 60)
print("SPLIT SUMMARY")
print("=" * 60)

print(
    "Training Images  :",
    len(train_ids)
)

print(
    "Validation Images:",
    len(validation_ids)
)

print(
    "Overlap          :",
    len(overlap)
)

assert len(overlap) == 0, \
    "Data leakage detected!"

print("\nTraining Distribution")

print(
    df_train_balanced["class"]
    .value_counts()
)

print("\nValidation Distribution")

print(
    df_validation["class"]
    .value_counts()
)

CREATING TRAIN / VALIDATION SPLITS
Ambiguous Images: 6
Clean Images: 5523

Saved splits:
/content/drive/MyDrive/TinyBayes/run_2000/train_2000_unique_images.csv
/content/drive/MyDrive/TinyBayes/run_2000/val_500_unique_images.csv

SPLIT SUMMARY
Training Images  : 2000
Validation Images: 500
Overlap          : 0

Training Distribution
class
cssvd          667
healthy        667
anthracnose    666
Name: count, dtype: int64

Validation Distribution
class
healthy        167
cssvd          167
anthracnose    166
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 4
# PREPARE DATASETS AND DATALOADERS
# ============================================================

BATCH_SIZE = 32
NUM_WORKERS = 2
IMAGE_SIZE = 224

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 60)
print("DEVICE")
print("=" * 60)

print(DEVICE)

# ------------------------------------------------------------
# Class Mapping
# ------------------------------------------------------------

CLASS_TO_INDEX = {
    cls: idx
    for idx, cls in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    idx: cls
    for cls, idx in CLASS_TO_INDEX.items()
}

print("\n" + "=" * 60)
print("CLASS MAPPING")
print("=" * 60)

print(CLASS_TO_INDEX)

# ------------------------------------------------------------
# Image Transformations
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

validation_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class CocoaDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir,
        transform
    ):

        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):

        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index
    ):

        row = self.dataframe.iloc[index]

        image_path = os.path.join(
            self.image_dir,
            row["Image_ID"]
        )

        image = cv2.imread(
            image_path
        )

        if image is None:

            raise FileNotFoundError(
                image_path
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        image = self.transform(
            image
        )

        label = CLASS_TO_INDEX[
            row["class"]
        ]

        return image, label

# ------------------------------------------------------------
# Datasets
# ------------------------------------------------------------

train_dataset = CocoaDataset(
    dataframe=df_train_balanced,
    image_dir=IMAGE_DIR_TRAIN,
    transform=train_transform
)

validation_dataset = CocoaDataset(
    dataframe=df_validation,
    image_dir=IMAGE_DIR_TRAIN,
    transform=validation_transform
)

# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print(
    "Training Images  :",
    len(train_dataset)
)

print(
    "Validation Images:",
    len(validation_dataset)
)

print(
    "Training Batches :",
    len(train_loader)
)

print(
    "Validation Batches:",
    len(validation_loader)
)

images, labels = next(
    iter(train_loader)
)

print("\n" + "=" * 60)
print("FIRST BATCH")
print("=" * 60)

print(
    "Images Shape :",
    images.shape
)

print(
    "Labels Shape :",
    labels.shape
)

print(
    "Image dtype  :",
    images.dtype
)

print(
    "Label dtype  :",
    labels.dtype
)

print(
    "Image Range  :",
    (
        float(images.min()),
        float(images.max())
    )
)

print(
    "Unique Labels:",
    torch.unique(labels).tolist()
)

DEVICE
cpu

CLASS MAPPING
{'anthracnose': 0, 'cssvd': 1, 'healthy': 2}

DATASET SUMMARY
Training Images  : 2000
Validation Images: 500
Training Batches : 63
Validation Batches: 16

FIRST BATCH
Images Shape : torch.Size([32, 3, 224, 224])
Labels Shape : torch.Size([32])
Image dtype  : torch.float32
Label dtype  : torch.int64
Image Range  : (-2.1179039478302, 2.640000104904175)
Unique Labels: [0, 1, 2]


In [ ]:
# ============================================================
# CELL 5
# FINE-TUNE MOBILENETV3-SMALL
# ============================================================

EPOCHS = 10
LEARNING_RATE = 1e-4

MODEL_PATH = os.path.join(
    SAVE_DIR,
    "mobilenet_v3_small_finetuned.pth"
)

HISTORY_PATH = os.path.join(
    SAVE_DIR,
    "training_history.csv"
)

# ============================================================
# BUILD MODEL
# ============================================================

model = mobilenet_v3_small(
    weights=MobileNet_V3_Small_Weights.DEFAULT
)

in_features = model.classifier[3].in_features

model.classifier[3] = nn.Linear(
    in_features,
    NUM_CLASSES
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# ============================================================
# LOAD EXISTING MODEL
# ============================================================

if os.path.exists(MODEL_PATH):

    print("=" * 60)
    print("LOADING EXISTING MODEL")
    print("=" * 60)

    model.load_state_dict(
        torch.load(
            MODEL_PATH,
            map_location=DEVICE
        )
    )

    history = pd.read_csv(
        HISTORY_PATH
    )

    display(history)

# ============================================================
# TRAIN MODEL
# ============================================================

else:

    print("=" * 60)
    print("TRAINING MOBILENETV3-SMALL")
    print("=" * 60)

    history = []

    best_accuracy = 0.0

    for epoch in range(EPOCHS):

        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in tqdm(train_loader):

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item() *
                labels.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

        train_loss = (
            running_loss / total
        )

        train_accuracy = (
            correct / total
        )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        model.eval()

        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in validation_loader:

                images = images.to(
                    DEVICE
                )

                labels = labels.to(
                    DEVICE
                )

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

                running_loss += (
                    loss.item() *
                    labels.size(0)
                )

                predictions = outputs.argmax(
                    dim=1
                )

                correct += (
                    predictions == labels
                ).sum().item()

                total += labels.size(0)

        validation_loss = (
            running_loss / total
        )

        validation_accuracy = (
            correct / total
        )

        history.append({

            "epoch":
                epoch + 1,

            "train_loss":
                train_loss,

            "train_accuracy":
                train_accuracy,

            "validation_loss":
                validation_loss,

            "validation_accuracy":
                validation_accuracy

        })

        print(

            f"Epoch {epoch+1:02d}/{EPOCHS} | "

            f"Train Loss: {train_loss:.4f} | "

            f"Train Acc: {train_accuracy:.4f} | "

            f"Val Loss: {validation_loss:.4f} | "

            f"Val Acc: {validation_accuracy:.4f}"

        )

        if validation_accuracy > best_accuracy:

            best_accuracy = (
                validation_accuracy
            )

            torch.save(
                model.state_dict(),
                MODEL_PATH
            )

    history = pd.DataFrame(
        history
    )

    history.to_csv(
        HISTORY_PATH,
        index=False
    )

    print("\nBest Validation Accuracy:",
          f"{best_accuracy:.4f}")

    print("\nSaved Model:")
    print(MODEL_PATH)

    print("\nSaved History:")
    print(HISTORY_PATH)

    display(history)

# ============================================================
# LOAD BEST MODEL
# ============================================================

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=DEVICE
    )
)

model.eval()

print("\nModel ready for feature extraction.")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 315MB/s]


TRAINING MOBILENETV3-SMALL


100%|██████████| 63/63 [10:12<00:00,  9.73s/it]


Epoch 01/10 | Train Loss: 0.8584 | Train Acc: 0.6460 | Val Loss: 0.7109 | Val Acc: 0.7120


...

Epoch 09/10 | Train Loss: 0.1303 | Train Acc: 0.9560 | Val Loss: 0.3602 | Val Acc: 0.8840


100%|██████████| 63/63 [03:00<00:00,  2.86s/it]


Epoch 10/10 | Train Loss: 0.1186 | Train Acc: 0.9605 | Val Loss: 0.3998 | Val Acc: 0.8720

Best Validation Accuracy: 0.8860

Saved Model:
/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_v3_small_finetuned.pth

Saved History:
/content/drive/MyDrive/TinyBayes/run_2000/training_history.csv


,epoch,train_loss,train_accuracy,validation_loss,validation_accuracy
0,1,0.858400,0.6460,0.710933,0.712
1,2,0.525725,0.7885,0.541764,0.788
2,3,0.405392,0.8485,0.464598,0.830
3,4,0.334458,0.8685,0.413177,0.860
4,5,0.275331,0.8980,0.395483,0.858
5,6,0.225836,0.9210,0.371123,0.866
6,7,0.213267,0.9285,0.380001,0.872
7,8,0.173843,0.9360,0.372609,0.886
8,9,0.130284,0.9560,0.360244,0.884
9,10,0.118576,0.9605,0.399842,0.872



Model ready for feature extraction.


In [ ]:
torch.save({
    "epoch": epoch + 1,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "validation_accuracy": validation_accuracy,
}, MODEL_PATH)

In [ ]:
# ============================================================
# CELL 5.5
# CREATE JACOBI-DMR EVALUATION SET
# ============================================================

JACOBI_TEST_SIZE = 500

JACOBI_TEST_PATH = os.path.join(
    SAVE_DIR,
    "jacobi_test_500_unique_images.csv"
)

# ============================================================
# LOAD EXISTING TEST SET
# ============================================================

if os.path.exists(JACOBI_TEST_PATH):

    print("=" * 60)
    print("LOADING EXISTING JACOBI TEST SET")
    print("=" * 60)

    df_jacobi_test = pd.read_csv(
        JACOBI_TEST_PATH
    )

# ============================================================
# CREATE NEW TEST SET
# ============================================================

else:

    print("=" * 60)
    print("CREATING JACOBI TEST SET")
    print("=" * 60)

    # --------------------------------------------------------
    # Remove ambiguous images
    # --------------------------------------------------------

    image_class_count = (
        df_train
        .groupby("Image_ID")["class"]
        .nunique()
    )

    ambiguous_images = set(
        image_class_count[
            image_class_count > 1
        ].index
    )

    clean_df = df_train[
        ~df_train["Image_ID"].isin(
            ambiguous_images
        )
    ].copy()

    unique_images = (
        clean_df[
            ["Image_ID", "class"]
        ]
        .drop_duplicates("Image_ID")
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Remove train + MobileNet validation images
    # --------------------------------------------------------

    used_images = set(
        df_train_balanced["Image_ID"]
    ).union(
        set(df_validation["Image_ID"])
    )

    remaining_images = unique_images[
        ~unique_images["Image_ID"].isin(
            used_images
        )
    ]

    print(
        "Remaining Images:",
        len(remaining_images)
    )

    # --------------------------------------------------------
    # Balanced sampling
    # --------------------------------------------------------

    test_per_class = (
        JACOBI_TEST_SIZE //
        NUM_CLASSES
    )

    test_parts = []

    for cls in CLASS_NAMES:

        cls_images = remaining_images[
            remaining_images["class"] == cls
        ]

        sampled = cls_images.sample(
            n=test_per_class,
            random_state=SEED
        )

        test_parts.append(
            sampled
        )

    df_jacobi_test = (
        pd.concat(test_parts)
        .sample(
            frac=1,
            random_state=SEED
        )
        .reset_index(drop=True)
    )

    remaining = (
        JACOBI_TEST_SIZE -
        len(df_jacobi_test)
    )

    if remaining > 0:

        pool = remaining_images[
            ~remaining_images["Image_ID"].isin(
                df_jacobi_test["Image_ID"]
            )
        ]

        extra = pool.sample(
            n=remaining,
            random_state=SEED
        )

        df_jacobi_test = (
            pd.concat(
                [
                    df_jacobi_test,
                    extra
                ]
            )
            .sample(
                frac=1,
                random_state=SEED
            )
            .reset_index(drop=True)
        )

    df_jacobi_test.to_csv(
        JACOBI_TEST_PATH,
        index=False
    )

# ============================================================
# VERIFY
# ============================================================

train_ids = set(df_train_balanced["Image_ID"])
val_ids = set(df_validation["Image_ID"])
test_ids = set(df_jacobi_test["Image_ID"])

print("\n" + "=" * 60)
print("JACOBI TEST SUMMARY")
print("=" * 60)

print(
    "Test Images:",
    len(test_ids)
)

print(
    "Train Overlap:",
    len(train_ids & test_ids)
)

print(
    "Validation Overlap:",
    len(val_ids & test_ids)
)

assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0

print("\nClass Distribution")

print(
    df_jacobi_test["class"]
    .value_counts()
)

CREATING JACOBI TEST SET
Remaining Images: 3023

JACOBI TEST SUMMARY
Test Images: 500
Train Overlap: 0
Validation Overlap: 0

Class Distribution
class
healthy        167
cssvd          167
anthracnose    166
Name: count, dtype: int64


In [ ]:
!pip install ai-edge-torch

INFO: pip is looking at multiple versions of ai-edge-litert[model-utils] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.8/575.8 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.3/419.3 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 25.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      S

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
print(checkpoint.keys())

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'validation_accuracy'])


In [ ]:
# ============================================================
# CELL 6
# LOAD FINE-TUNED MOBILENET AS FEATURE EXTRACTOR
# ============================================================

import torch
import torch.nn as nn

MODEL_PATH = os.path.join(
    SAVE_DIR,
    "mobilenet_v3_small_finetuned.pth"
)

FEATURE_DIM = 576

# ------------------------------------------------------------
# Build MobileNet Architecture
# ------------------------------------------------------------

model = mobilenet_v3_small(weights=None)

model.classifier[3] = nn.Linear(
    model.classifier[3].in_features,
    NUM_CLASSES
)

# ------------------------------------------------------------
# Load Checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(DEVICE)
model.eval()

print("=" * 60)
print("Fine-tuned MobileNet Loaded")
print("=" * 60)
print(f"Checkpoint Epoch      : {checkpoint['epoch']}")
print(f"Validation Accuracy   : {checkpoint['validation_accuracy']:.4f}")

# ------------------------------------------------------------
# Create Feature Extractor
# ------------------------------------------------------------

class MobileNetFeatureExtractor(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.features = backbone.features
        self.avgpool = backbone.avgpool

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x


feature_extractor = MobileNetFeatureExtractor(model).to(DEVICE)
feature_extractor.eval()

# ------------------------------------------------------------
# Freeze Parameters
# ------------------------------------------------------------

for param in feature_extractor.parameters():
    param.requires_grad = False

# ------------------------------------------------------------
# Verify Feature Dimension
# ------------------------------------------------------------

dummy = torch.randn(1, 3, 224, 224).to(DEVICE)

with torch.no_grad():
    features = feature_extractor(dummy)

print("\n" + "=" * 60)
print("Feature Extractor Verification")
print("=" * 60)
print("Output Shape :", tuple(features.shape))

assert features.shape[1] == FEATURE_DIM, \
    f"Expected {FEATURE_DIM} features but got {features.shape[1]}"

print(f"Feature Dimension : {FEATURE_DIM}")
print("Feature extractor is ready.")

Fine-tuned MobileNet Loaded
Checkpoint Epoch      : 10
Validation Accuracy   : 0.8720

Feature Extractor Verification
Output Shape : (1, 576)
Feature Dimension : 576
Feature extractor is ready.


In [ ]:
# ============================================================
# CELL 7
# EXTRACT FEATURES FOR JACOBI-DMR
# ============================================================

from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import numpy as np

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class FeatureExtractionDataset(Dataset):

    def __init__(self, dataframe, image_dir, transform):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = os.path.join(
            self.image_dir,
            row["Image_ID"]
        )

        # Read image with OpenCV just like the training dataset
        image = cv2.imread(image_path)

        if image is None:
            raise FileNotFoundError(image_path)

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        image = self.transform(image)

        label = CLASS_TO_INDEX[row["class"]]

        return image, label


# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 32

train_feature_loader = DataLoader(
    FeatureExtractionDataset(
        df_train_balanced,
        IMAGE_DIR_TRAIN,
        validation_transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

jacobi_test_loader = DataLoader(
    FeatureExtractionDataset(
        df_jacobi_test,
        IMAGE_DIR_TRAIN,
        validation_transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# Feature Extraction Function
# ------------------------------------------------------------

def extract_features(loader):

    feature_extractor.eval()

    features_list = []
    labels_list = []

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(DEVICE)

            features = feature_extractor(images)

            features_list.append(
                features.cpu().numpy()
            )

            labels_list.append(
                labels.numpy()
            )

    features = np.concatenate(features_list, axis=0)
    labels = np.concatenate(labels_list, axis=0)

    return features, labels


# ------------------------------------------------------------
# Extract Train Features
# ------------------------------------------------------------

print("=" * 60)
print("Extracting Training Features")
print("=" * 60)

train_features, train_labels = extract_features(
    train_feature_loader
)

# ------------------------------------------------------------
# Extract Test Features
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Extracting Jacobi Test Features")
print("=" * 60)

jacobi_test_features, jacobi_test_labels = extract_features(
    jacobi_test_loader
)

# ------------------------------------------------------------
# Save Features
# ------------------------------------------------------------

np.save(
    os.path.join(SAVE_DIR, "train_features.npy"),
    train_features
)

np.save(
    os.path.join(SAVE_DIR, "train_labels.npy"),
    train_labels
)

np.save(
    os.path.join(SAVE_DIR, "jacobi_test_features.npy"),
    jacobi_test_features
)

np.save(
    os.path.join(SAVE_DIR, "jacobi_test_labels.npy"),
    jacobi_test_labels
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FEATURE EXTRACTION COMPLETE")
print("=" * 60)

print(f"Train Features Shape       : {train_features.shape}")
print(f"Train Labels Shape         : {train_labels.shape}")

print(f"Jacobi Test Features Shape : {jacobi_test_features.shape}")
print(f"Jacobi Test Labels Shape   : {jacobi_test_labels.shape}")

print("\nSaved Files")
print("- train_features.npy")
print("- train_labels.npy")
print("- jacobi_test_features.npy")
print("- jacobi_test_labels.npy")

Extracting Training Features


100%|██████████| 63/63 [02:35<00:00,  2.47s/it]



Extracting Jacobi Test Features


100%|██████████| 16/16 [03:02<00:00, 11.41s/it]


FEATURE EXTRACTION COMPLETE
Train Features Shape       : (2000, 576)
Train Labels Shape         : (2000,)
Jacobi Test Features Shape : (500, 576)
Jacobi Test Labels Shape   : (500,)

Saved Files
- train_features.npy
- train_labels.npy
- jacobi_test_features.npy
- jacobi_test_labels.npy


In [ ]:
# ============================================================
# CELL 8
# LOAD FEATURE VECTORS
# ============================================================

import numpy as np

train_features = np.load(
    os.path.join(SAVE_DIR, "train_features.npy")
)

train_labels = np.load(
    os.path.join(SAVE_DIR, "train_labels.npy")
)

test_features = np.load(
    os.path.join(SAVE_DIR, "jacobi_test_features.npy")
)

test_labels = np.load(
    os.path.join(SAVE_DIR, "jacobi_test_labels.npy")
)

print("=" * 60)
print("FEATURE DATA")
print("=" * 60)

print("Train Features :", train_features.shape)
print("Train Labels   :", train_labels.shape)

print("Test Features  :", test_features.shape)
print("Test Labels    :", test_labels.shape)

print("\nFeature dtype :", train_features.dtype)

print(
    "\nFeature Range:",
    train_features.min(),
    train_features.max()
)

FEATURE DATA
Train Features : (2000, 576)
Train Labels   : (2000,)
Test Features  : (500, 576)
Test Labels    : (500,)

Feature dtype : float32

Feature Range: -0.31589463 5.150978


In [ ]:
# ============================================================
# CELL 9
# JACOBI-DMR CLASSIFIER
# ============================================================

import time
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("=" * 60)
print("TRAINING JACOBI-DMR")
print("=" * 60)

start_time = time.time()

# ------------------------------------------------------------
# Prepare Data
# ------------------------------------------------------------

X_train = train_features.astype(np.float64)
X_test = test_features.astype(np.float64)

y_train = train_labels
y_test = test_labels

# ------------------------------------------------------------
# One-Hot Labels
# ------------------------------------------------------------

y_one_hot = pd.get_dummies(y_train)

# Ensure column order matches label indices
y_one_hot = y_one_hot.reindex(
    columns=range(NUM_CLASSES),
    fill_value=0
)

# ------------------------------------------------------------
# Jacobi Prior
# ------------------------------------------------------------

a = b = 1 / len(X_train)
k = 1

etas = []

for c in range(NUM_CLASSES):

    y = y_one_hot[c].values

    eta = np.log(
        (y + a) /
        (1 + k * b)
    )

    etas.append(eta)

# ------------------------------------------------------------
# Closed-form Coefficients
# ------------------------------------------------------------

XTX_inv = np.linalg.pinv(
    X_train.T @ X_train
)

betas = []

for eta in etas:

    beta = (
        XTX_inv
        @
        (X_train.T @ eta)
    )

    betas.append(beta)

# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

scores = np.column_stack([
    np.exp(X_test @ beta)
    for beta in betas
])

y_pred = np.argmax(
    scores,
    axis=1
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

elapsed = time.time() - start_time

print(f"\nAccuracy : {accuracy:.4f}")
print(f"Time     : {elapsed:.4f} seconds")

print("\nClassification Report\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES
    )
)

print("\nConfusion Matrix\n")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

TRAINING JACOBI-DMR

Accuracy : 0.8800
Time     : 1.4291 seconds

Classification Report

              precision    recall  f1-score   support

 anthracnose       0.91      0.90      0.90       166
       cssvd       0.83      0.93      0.88       167
     healthy       0.91      0.81      0.86       167

    accuracy                           0.88       500
   macro avg       0.88      0.88      0.88       500
weighted avg       0.88      0.88      0.88       500


Confusion Matrix

[[149   9   8]
 [  6 155   6]
 [  9  22 136]]


In [ ]:
# ============================================================
# CELL 10
# SAVE JACOBI-DMR MODEL
# ============================================================

import json
import numpy as np

# ------------------------------------------------------------
# Save Beta Matrix
# ------------------------------------------------------------

beta_matrix = np.stack(betas)

beta_path = os.path.join(
    SAVE_DIR,
    "jacobi_betas.npy"
)

np.save(
    beta_path,
    beta_matrix
)

# ------------------------------------------------------------
# Save Class Names
# ------------------------------------------------------------

class_path = os.path.join(
    SAVE_DIR,
    "class_names.json"
)

with open(class_path, "w") as f:
    json.dump(
        CLASS_NAMES,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

loaded_beta = np.load(beta_path)

print("=" * 60)
print("JACOBI MODEL SAVED")
print("=" * 60)

print("Beta Matrix Shape :", loaded_beta.shape)
print("Feature Dimension :", loaded_beta.shape[1])
print("Number of Classes :", loaded_beta.shape[0])

print("\nSaved Files")
print(beta_path)
print(class_path)

JACOBI MODEL SAVED
Beta Matrix Shape : (3, 576)
Feature Dimension : 576
Number of Classes : 3

Saved Files
/content/drive/MyDrive/TinyBayes/run_2000/jacobi_betas.npy
/content/drive/MyDrive/TinyBayes/run_2000/class_names.json


In [ ]:
# ============================================================
# CELL 11
# EXPORT FEATURE EXTRACTOR TO TORCHSCRIPT
# ============================================================

import torch
import torch.nn as nn

# ------------------------------------------------------------
# Rebuild Feature Extractor
# ------------------------------------------------------------

class MobileNetFeatureExtractor(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.features = backbone.features
        self.avgpool = backbone.avgpool

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x

feature_extractor = MobileNetFeatureExtractor(model)
feature_extractor.eval()

# ------------------------------------------------------------
# Example Input
# ------------------------------------------------------------

example = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

scripted_model = torch.jit.trace(
    feature_extractor,
    example
)

TORCHSCRIPT_PATH = os.path.join(
    SAVE_DIR,
    "mobilenet_feature_extractor.pt"
)

scripted_model.save(TORCHSCRIPT_PATH)

print("=" * 60)
print("TORCHSCRIPT MODEL SAVED")
print("=" * 60)
print(TORCHSCRIPT_PATH)

TORCHSCRIPT MODEL SAVED
/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_feature_extractor.pt


In [ ]:
# ============================================================
# CELL 12
# VERIFY FEATURE EQUIVALENCE
# ============================================================

import torch

# ------------------------------------------------------------
# Load TorchScript Model
# ------------------------------------------------------------

torchscript_model = torch.jit.load(TORCHSCRIPT_PATH)
torchscript_model.eval()

feature_extractor.eval()

# ------------------------------------------------------------
# Compare Feature Outputs
# ------------------------------------------------------------

max_difference = 0.0
mean_difference = 0.0
count = 0

with torch.no_grad():

    for images, _ in jacobi_test_loader:

        # Original PyTorch model
        pytorch_features = feature_extractor(images)

        # TorchScript model
        torchscript_features = torchscript_model(images)

        # Absolute difference
        diff = torch.abs(
            pytorch_features -
            torchscript_features
        )

        max_difference = max(
            max_difference,
            diff.max().item()
        )

        mean_difference += diff.mean().item()

        count += 1

mean_difference /= count

print("=" * 60)
print("FEATURE COMPARISON")
print("=" * 60)

print(f"Maximum Difference : {max_difference:.10f}")
print(f"Average Difference : {mean_difference:.10f}")

if max_difference < 1e-6:
    print("\n✅ TorchScript reproduces the PyTorch feature extractor.")
else:
    print("\n⚠️ TorchScript output differs from the original model.")

FEATURE COMPARISON
Maximum Difference : 0.0000000000
Average Difference : 0.0000000000

✅ TorchScript reproduces the PyTorch feature extractor.


In [ ]:
# ============================================================
# CELL 11
# EXPORT JACOBI COEFFICIENTS FOR ANDROID
# ============================================================

import os
import numpy as np

beta_matrix = np.stack(betas).astype(np.float32)

bin_path = os.path.join(
    SAVE_DIR,
    "jacobi_betas.bin"
)

beta_matrix.tofile(bin_path)

print("=" * 60)
print("ANDROID COEFFICIENT FILE CREATED")
print("=" * 60)
print("Shape :", beta_matrix.shape)
print("Saved :", bin_path)

ANDROID COEFFICIENT FILE CREATED
Shape : (3, 576)
Saved : /content/drive/MyDrive/TinyBayes/run_2000/jacobi_betas.bin
